# Fine-tune Ouroboros A/C on real-style synthetic drawings and labelled R-groups

Branch: `feat/rgroups-aug-oct29`. Choose a GPU runtime first. This notebook mounts Drive,
clones the branch, installs the repository's Python 3.11 training stack, fetches released
weights and the seeded 100-per-source evaluation subset, builds up to 10,000 synthetic training
examples (50% attempted R-group/abbreviation substitution), checks leakage, fine-tunes A
and C, and scores the validation-selected checkpoints on the **held-out** real subset.
No real evaluation image or label is used for training or checkpoint selection. Optional
public real training data is deliberately omitted; add it only after checking image/source
identity and molecular overlap with `scripts/check_leakage.py`.

All runs/checkpoints/reports are saved under
`/content/drive/MyDrive/ouroboros-ocsr_runs/real_aug_oct29/`. Keep this run name to resume;
change it when changing data, vocabulary, architecture, or scheduler settings.


In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/ouroboros-ocsr')
BRANCH = 'feat/rgroups-aug-oct29'
DRIVE_ROOT = Path('/content/drive/MyDrive/ouroboros-ocsr_runs/real_aug_oct29')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ARMS = ['A', 'C']  # use ['A'] for a shorter first experiment
N_TRAIN, N_VAL, N_TEST = 10000, 300, 200
N_VALIDATION = 200
STEPS, BATCH = 2000, 8
PY = '/content/venv-real-aug/bin/python'
os.environ.update(OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
                  MPLBACKEND='Agg', MPLCONFIGDIR='/tmp/mpl-real-aug')

def shell(args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=str(cwd) if cwd else None, check=True)

def run(*args):
    shell([PY, *args], cwd=REPO)

def run_python(source):
    run('-c', source)

shell(['nvidia-smi'])
if not REPO.exists():
    shell(['git', 'clone', 'https://github.com/yaniguan/ouroboros-ocsr.git', REPO])
shell(['git', 'fetch', 'origin', BRANCH], cwd=REPO)
shell(['git', 'checkout', BRANCH], cwd=REPO)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
(DRIVE_ROOT / 'source_revision.txt').write_text(revision + '\n')
print('Source:', revision)


## Install the tested training dependencies

The kernel may use Python 3.13; project commands use a separate Python 3.11 environment,
as in the repository's other Colab notebooks. `gfortran` is needed by `py3nj`/`escnn`.
The full Linux CUDA pins come from `requirements.txt`; the editable install uses
`--no-deps`. Setup and downloads require internet access and several GB of disk.


In [ ]:
if not shutil.which('gfortran'):
    shell(['apt-get', '-qq', 'update'])
    shell(['apt-get', '-qq', 'install', '-y', 'gfortran'])
shell([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
uv = shutil.which('uv')
assert uv, 'uv installation did not put uv on PATH'
if not Path(PY).exists():
    shell([uv, 'venv', '--python', '3.11', '--python-preference', 'only-managed',
           str(Path(PY).parent.parent)])
shell([uv, 'pip', 'install', '--python', PY, '-r', REPO / 'requirements.txt'])
shell([uv, 'pip', 'install', '--python', PY, '--no-deps', '-e', REPO])
shell([uv, 'pip', 'install', '--python', PY, 'pyarrow'])
run_python("import torch, rdkit; print(torch.__version__, rdkit.__version__); "
           "assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name())")
with (DRIVE_ROOT / 'environment.txt').open('w') as stream:
    subprocess.run([uv, 'pip', 'freeze', '--python', PY], stdout=stream, check=True)
run('-m', 'pytest', '-q', 'tests/test_rgroups_aug.py')
for arm in ARMS:
    run('-m', 'ouroboros.weights', 'fetch', '--arm', arm)


## Fetch and ingest the evaluation subset; keep it isolated

The evaluation subset uses `seed=0`, USPTO/UOB/ACS, 100 images each, matching the local
experiment. The public HF source defaults to `main`; its contents can change. This
notebook records the sampled manifest hash and downloaded parquet hashes. Compare these
with the local worklog before claiming identical-data reproduction. Expected current
ingestion is 298/300; 20 previously rejected labels now parse. The two malformed
annotations are logged and excluded; the scoring denominator must be reported.


In [ ]:
run('scripts/fetch_real_subset.py', '--sets', 'USPTO', 'UOB', 'ACS', '--n', '100',
    '--seed', '0', '--out', 'real_data/subset100')
run('scripts/ingest_real.py', '--manifest', 'real_data/subset100/manifest.csv',
    '--out', 'real/subset100')
shutil.copytree(REPO / 'real/subset100', DRIVE_ROOT / 'eval_shards', dirs_exist_ok=True)
shutil.copy2(REPO / 'real_data/subset100/manifest.csv', DRIVE_ROOT / 'eval_manifest.csv')
run_python("""
import hashlib, json
from pathlib import Path
paths = [Path('real_data/subset100/manifest.csv'), *Path('real_data/subset100/parquet').glob('*.parquet')]
receipt = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
Path('real/subset100/source_hashes.json').write_text(json.dumps(receipt, indent=2))
print(json.dumps(receipt, indent=2))
""")
shutil.copy2(REPO / 'real/subset100/source_hashes.json', DRIVE_ROOT / 'eval_source_hashes.json')
run('scripts/check_leakage.py', '--eval', 'real=real/subset100:test',
    '--dump-eval-keys', '/content/eval_keys.txt')


## Generate synthetic fine-tuning data

Source: the repository's checksum-verified ZINC250k list. Exclude real-evaluation molecule
identities before composition, preserve molecule-disjoint train/val/test splits, then
render with varied fonts/bond widths/styles, abbreviation collapse, and scan effects.
Substitution is attempted with probability 0.5; molecules without a safe pendant group
remain unchanged. Abbreviations collapse their exact chemical fragments; R/R1–R4/X
represent unspecified substituents. A labelled abbreviation is scored as that label,
not as its expanded chemical graph. All output labels come from the drawn molecule.

Reuse matching Drive artifacts across disconnects. The generator refuses mismatched
settings/checksums; use a new dataset/run directory when changing the recipe.


In [ ]:
DATA = Path('/content/data/real-aug')
if (DRIVE_ROOT / 'dataset').exists() and not DATA.exists():
    shutil.copytree(DRIVE_ROOT / 'dataset', DATA)
t0 = time.perf_counter()
run('scripts/build_dataset.py', '--out', DATA, '--sources', 'zinc250k',
    '--cache', '/content/data_cache', '--train-size', str(N_TRAIN),
    '--val-size', str(N_VAL), '--test-size', str(N_TEST), '--workers', '2',
    '--seed', '29', '--exclude-keys', '/content/eval_keys.txt',
    '--rgroup-probability', '0.5', '--real-style', '--max-tokens', '128')
print('Dataset generation seconds:', time.perf_counter() - t0)
run('scripts/check_leakage.py', '--eval', 'real=real/subset100:test',
    f'val={DATA}/shards:val', f'test={DATA}/shards:test',
    '--train', f'synthetic={DATA}/shards:train')
run('scripts/benchmark_real_aug.py', '--n', '200', '--out', DRIVE_ROOT / 'augmentation')
shutil.copytree(DATA, DRIVE_ROOT / 'dataset', dirs_exist_ok=True)


## Prepare append-only vocabulary and configs from the released architectures

Released vocabularies stay unchanged during inference. For fine-tuning only, append label
and newly observed synthetic tokens, preserve every original token ID, and copy all old
embedding/output rows. New rows use the normal model initialization. Optimizer/scheduler
start fresh; subsequent runs in the same output directory resume `ckpt/last.pt` exactly.
Overlength labels are dropped and counted during rendering; this cell verifies the remaining labels.


In [ ]:
prepare = r"""
import json, torch, yaml
from pathlib import Path
from ouroboros.data.loader import ShardDataset
from ouroboros.decode.tokenizer import SmilesTokenizer, tokenize
from ouroboros.train.config import load_config
DATA = Path('/content/data/real-aug')
ARMS = __ARMS__
DRIVE_ROOT = Path(__DRIVE__)
configs = {}
for arm in ARMS:
    ckpt = Path('checkpoints') / f'ouroboros-{arm}.pt'
    state = torch.load(ckpt, map_location='cpu', weights_only=False)
    tok = SmilesTokenizer(state['vocab'])
    tok.add_label_tokens()
    for split in ('train', 'val', 'test'):
        ds = ShardDataset(DATA / 'shards', split)
        for i in range(len(ds)):
            label = ds.meta(i)['smiles']
            tok.extend(tokenize(label))
            assert len(tok.encode(label)) - 1 <= state['config']['model']['decoder']['max_len'], label
    vocab = DATA / f'vocab-{arm}.json'
    tok.save(vocab)
    cfg = load_config('configs/finetune-real-aug.yaml')
    cfg['model'] = state['config']['model']
    cfg['data'].update(vocab=str(vocab), synthetic_root=str(DATA / 'shards'),
                       image_size=state['config']['data']['image_size'],
                       synthetic_max_samples=__NTRAIN__, num_workers=2)
    cfg['train'].update(init_checkpoint=str(ckpt.resolve()), steps=__STEPS__, batch_size=__BATCH__)
    cfg['validation'].update(root=str(DATA / 'shards'), samples=__NVAL__, batch_size=__BATCH__)
    # Native bf16 only; T4 uses fp32, supported by the existing Trainer without a scaler.
    cfg['train']['amp'] = 'bf16' if torch.cuda.is_bf16_supported() else 'none'
    path = DRIVE_ROOT / f'finetune-{arm}.yaml'
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    configs[arm] = str(path)
    del state
(DRIVE_ROOT / 'configs.json').write_text(json.dumps(configs, indent=2))
"""
for marker, value in {'__ARMS__': repr(ARMS), '__DRIVE__': repr(str(DRIVE_ROOT)),
                      '__NTRAIN__': str(N_TRAIN), '__NVAL__': str(N_VALIDATION),
                      '__STEPS__': str(STEPS), '__BATCH__': str(BATCH)}.items():
    prepare = prepare.replace(marker, value)
run_python(prepare)
CONFIGS = json.loads((DRIVE_ROOT / 'configs.json').read_text())
shutil.copytree(DATA, DRIVE_ROOT / 'dataset', dirs_exist_ok=True)


## Runtime estimate and capacity check

**Planning estimates, not measured GPU results:** allow roughly 30–120 minutes on a T4,
20–90 minutes on an L4, or 15–60 minutes on an A100 for setup, 10k generation and both
2,000-step arms with validation; early stopping can shorten training. These deliberately
broad budgets use measured local CPU generation (~68.9 labelled renders/s at 384px;
10,500 images ≈2.5 minutes excluding pool/layout diversity) and inference (A: 298 images
in 68.0s, B: 99.6s), plus a two-step CPU fine-tuning smoke run at batch 2 (3.34s including validation/checkpoint I/O),
and the repository's **unmeasured** A100 training assumptions in
`configs/compute_assumptions.yaml` (600/400 images/s at batch 64, bf16). Those assumptions
do not directly apply to batch 8, T4 fp32, validation decoding, Drive I/O, or installation.
There is no measured T4/L4/A100 speedup claim here. Use the capacity probe and first 200
training steps to replace these estimates; logs report actual throughput. C may need a
smaller batch on low-memory GPUs; change BATCH and use a new run name if the probe fails.


In [ ]:
for arm in ARMS:
    precision = 'fp32'  # safe worst-case memory probe, including T4
    run('scripts/benchmark_cuda.py', '--config', CONFIGS[arm],
        '--output', DRIVE_ROOT / f'capacity-{arm}.json', '--steps', '8', '--warmup', '3',
        '--batch', str(BATCH), '--sequence-length', '127', '--precision', precision)


## Fine-tune on GPU and evaluate the validation-selected checkpoint

Validation uses the fixed synthetic **val** split only. After each arm, the notebook runs
both the original release and the best fine-tuned checkpoint on real/subset100 with the
same prediction script and scoring harness. Reports include counts, validity, stereo-aware
exact match, flat exact, Tanimoto and source-specific breakdowns. No gain is assumed.
Scoring parity with arXiv:2608.09100 remains unverified. Dummy-containing molecules have
no InChIKey and receive false for that cross-check (even when exact labels match).


In [ ]:
for arm in ARMS:
    run_dir = DRIVE_ROOT / f'train-{arm}'
    t0 = time.perf_counter()
    run('-m', 'ouroboros.train', '--config', CONFIGS[arm], '--out', run_dir)
    print(arm, 'training wall seconds:', time.perf_counter() - t0)
    selection = r"""
import torch
from pathlib import Path
root = Path(__RUN__)
state = torch.load(root / 'ckpt/last.pt', map_location='cpu', weights_only=False)
best_step = state['early_stop']['best_step']
assert best_step is not None, 'No validation-selected checkpoint was produced'
selected = root / 'ckpt' / f'best-step-{best_step:08d}.pt'
assert selected.exists()
(root / 'selected_checkpoint.txt').write_text(str(selected))
print('Selected synthetic-validation best:', selected)
""".replace('__RUN__', repr(str(run_dir)))
    run_python(selection)
    selected = Path((run_dir / 'selected_checkpoint.txt').read_text())
    for stage, checkpoint in [('released', REPO / f'checkpoints/ouroboros-{arm}.pt'),
                               ('finetuned', selected)]:
        prefix = DRIVE_ROOT / f'{arm}-{stage}'
        run('scripts/predict_real.py', '--ckpt', checkpoint, '--shards', 'real/subset100',
            '--out', str(prefix) + '-preds.csv', '--batch-size', str(BATCH))
        run('scripts/eval_predictions.py', str(prefix) + '-preds.csv',
            '--json', str(prefix) + '-report.json', '--md', str(prefix) + '-report.md')


## Save and download results

Drive already holds configs, dataset, exact-resume checkpoints, logs, eval shards and
reports. The ZIP below includes reports, configs, provenance, logs and the selected
checkpoints, excluding large dataset shards and redundant training checkpoints.


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content/ouroboros-real-aug-results.zip')
selected_paths = set()
for arm in ARMS:
    marker = DRIVE_ROOT / f'train-{arm}/selected_checkpoint.txt'
    if marker.exists():
        selected_paths.add(Path(marker.read_text()))
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in DRIVE_ROOT.rglob('*'):
        if not path.is_file() or 'dataset' in path.relative_to(DRIVE_ROOT).parts or 'eval_shards' in path.relative_to(DRIVE_ROOT).parts:
            continue
        if path.suffix == '.zip':
            continue
        if path.suffix == '.pt' and path not in selected_paths:
            continue
        z.write(path, path.relative_to(DRIVE_ROOT))
shutil.copy2(archive, DRIVE_ROOT / archive.name)
print('Saved:', DRIVE_ROOT, 'ZIP bytes:', archive.stat().st_size)
files.download(str(archive))
